# Modeling

In [117]:
import pandas as pd
import numpy as np
import os
import json
import time
import joblib
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

required_files = [
    '../data/processed/X_train_tfidf.npz', '../data/processed/X_test_tfidf.npz',
    '../data/processed/y_train.csv', '../data/processed/y_test.csv'
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f'File berikut belum ada: {missing}\n'
        'Jalankan dulu 08_feature_engineering.ipynb (yang butuh output Tahap 6 dari Colab).'
    )

X_train = sp.load_npz('../data/processed/X_train_tfidf.npz')
X_test = sp.load_npz('../data/processed/X_test_tfidf.npz')
y_train = pd.read_csv('../data/processed/y_train.csv')['sentiment_label']
y_test = pd.read_csv('../data/processed/y_test.csv')['sentiment_label']

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'y_train distribusi:\n{y_train.value_counts()}')

X_train: (95022, 10000) | X_test: (23756, 10000)
y_train distribusi:
sentiment_label
negative    55266
neutral     22154
positive    17602
Name: count, dtype: int64


## Model 1: Logistic Regression

In [79]:
t0 = time.time()
model_lr = LogisticRegression(
    max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1, C=1.0
)
model_lr.fit(X_train, y_train)
train_time_lr = time.time() - t0

y_pred_lr = model_lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr, rec_lr, f1_lr, _ = precision_recall_fscore_support(y_test, y_pred_lr, average='macro', zero_division=0)

print(f'Waktu training: {train_time_lr:.2f}s')
print(f'Accuracy       : {acc_lr:.4f}')
print(f'Macro Precision: {prec_lr:.4f}')
print(f'Macro Recall   : {rec_lr:.4f}')
print(f'Macro F1       : {f1_lr:.4f}')
print()
print(classification_report(y_test, y_pred_lr, zero_division=0))

Waktu training: 9.20s
Accuracy       : 0.7198
Macro Precision: 0.6787
Macro Recall   : 0.7288
Macro F1       : 0.6932

              precision    recall  f1-score   support

    negative       0.89      0.70      0.78     13817
     neutral       0.60      0.75      0.67      5538
    positive       0.55      0.73      0.63      4401

    accuracy                           0.72     23756
   macro avg       0.68      0.73      0.69     23756
weighted avg       0.76      0.72      0.73     23756



## Model 2: Support Vector Machine (LinearSVC)

In [80]:
t0 = time.time()
model_svm = LinearSVC(class_weight='balanced', random_state=42, C=1.0, max_iter=5000)
model_svm.fit(X_train, y_train)
train_time_svm = time.time() - t0

y_pred_svm = model_svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm, rec_svm, f1_svm, _ = precision_recall_fscore_support(y_test, y_pred_svm, average='macro', zero_division=0)

print(f'Waktu training: {train_time_svm:.2f}s')
print(f'Accuracy       : {acc_svm:.4f}')
print(f'Macro Precision: {prec_svm:.4f}')
print(f'Macro Recall   : {rec_svm:.4f}')
print(f'Macro F1       : {f1_svm:.4f}')
print()
print(classification_report(y_test, y_pred_svm, zero_division=0))

Waktu training: 3.56s
Accuracy       : 0.7265
Macro Precision: 0.6799
Macro Recall   : 0.6993
Macro F1       : 0.6864

              precision    recall  f1-score   support

    negative       0.85      0.77      0.81     13817
     neutral       0.57      0.71      0.64      5538
    positive       0.62      0.62      0.62      4401

    accuracy                           0.73     23756
   macro avg       0.68      0.70      0.69     23756
weighted avg       0.74      0.73      0.73     23756



##Model 3: Multinomial Naive Bayes (Tambahan Pembanding)

In [81]:
t0 = time.time()
model_nb = MultinomialNB(alpha=0.5)
model_nb.fit(X_train, y_train)
train_time_nb = time.time() - t0

y_pred_nb = model_nb.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)
prec_nb, rec_nb, f1_nb, _ = precision_recall_fscore_support(y_test, y_pred_nb, average='macro', zero_division=0)

print(f'Waktu training: {train_time_nb:.2f}s')
print(f'Accuracy       : {acc_nb:.4f}')
print(f'Macro Precision: {prec_nb:.4f}')
print(f'Macro Recall   : {rec_nb:.4f}')
print(f'Macro F1       : {f1_nb:.4f}')
print()
print(classification_report(y_test, y_pred_nb, zero_division=0))

Waktu training: 0.23s
Accuracy       : 0.7169
Macro Precision: 0.7194
Macro Recall   : 0.5961
Macro F1       : 0.6289

              precision    recall  f1-score   support

    negative       0.72      0.91      0.80     13817
     neutral       0.69      0.48      0.57      5538
    positive       0.75      0.39      0.51      4401

    accuracy                           0.72     23756
   macro avg       0.72      0.60      0.63     23756
weighted avg       0.72      0.72      0.70     23756



## Validasi Tambahan: 5-Fold Cross-Validation pada Data Train


In [82]:
cv_results = {}
for name, model in [('Logistic Regression', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)),
                     ('SVM (LinearSVC)', LinearSVC(class_weight='balanced', random_state=42, max_iter=5000)),
                     ('Naive Bayes', MultinomialNB(alpha=0.5))]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
    cv_results[name] = {'mean_f1_macro': scores.mean(), 'std_f1_macro': scores.std()}
    print(f'{name:22s}: F1-macro = {scores.mean():.4f} (+/- {scores.std():.4f})')

Logistic Regression   : F1-macro = 0.6832 (+/- 0.0025)
SVM (LinearSVC)       : F1-macro = 0.6780 (+/- 0.0041)
Naive Bayes           : F1-macro = 0.6268 (+/- 0.0030)


## Perbandingan Performa Semua Model

In [83]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM (LinearSVC)', 'Naive Bayes'],
    'Accuracy': [acc_lr, acc_svm, acc_nb],
    'Macro Precision': [prec_lr, prec_svm, prec_nb],
    'Macro Recall': [rec_lr, rec_svm, rec_nb],
    'Macro F1': [f1_lr, f1_svm, f1_nb],
    'CV F1-macro (mean)': [cv_results['Logistic Regression']['mean_f1_macro'],
                            cv_results['SVM (LinearSVC)']['mean_f1_macro'],
                            cv_results['Naive Bayes']['mean_f1_macro']],
    'Training Time (s)': [train_time_lr, train_time_svm, train_time_nb],
}).sort_values('Macro F1', ascending=False).reset_index(drop=True)

comparison

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,CV F1-macro (mean),Training Time (s)
0,Logistic Regression,0.719776,0.678745,0.728791,0.693215,0.683153,9.204301
1,SVM (LinearSVC),0.726469,0.679931,0.699282,0.686417,0.677968,3.558238
2,Naive Bayes,0.716872,0.719416,0.596064,0.628866,0.626768,0.233239


In [84]:
best_model_name = comparison.iloc[0]['Model']
print(f'Model terbaik berdasarkan Macro F1 pada test set: {best_model_name}')
print()
print('Catatan: Macro F1 (bukan Accuracy) dipakai sebagai kriteria utama karena lebih adil')
print('ketika distribusi kelas sentimen tidak seimbang (tidak bias ke kelas mayoritas).')

Model terbaik berdasarkan Macro F1 pada test set: Logistic Regression

Catatan: Macro F1 (bukan Accuracy) dipakai sebagai kriteria utama karena lebih adil
ketika distribusi kelas sentimen tidak seimbang (tidak bias ke kelas mayoritas).


## Simpan Semua Model & Hasil Perbandingan

In [85]:
os.makedirs('models', exist_ok=True)

joblib.dump(model_lr, '../models/model_logistic_regression.pkl')
joblib.dump(model_svm, '../models/model_svm.pkl')
joblib.dump(model_nb, '../models/model_naive_bayes.pkl')

model_map = {'Logistic Regression': model_lr, 'SVM (LinearSVC)': model_svm, 'Naive Bayes': model_nb}
best_model = model_map[best_model_name]
joblib.dump(best_model, '../models/best_model.pkl')

with open('../models/best_model_name.txt', 'w') as f:
    f.write(best_model_name)

comparison.to_csv('../reports/model_comparison.csv', index=False)

modeling_summary = {
    'models_trained': ['Logistic Regression', 'SVM (LinearSVC)', 'Naive Bayes'],
    'best_model': best_model_name,
    'comparison_table': comparison.to_dict(orient='records'),
    'selection_criterion': 'Macro F1-score pada test set (robust terhadap class imbalance)',
}
with open('../reports/modeling_summary.json', 'w') as f:
    json.dump(modeling_summary, f, indent=2, default=str)

print('Model tersimpan di models/: model_logistic_regression.pkl, model_svm.pkl, model_naive_bayes.pkl, best_model.pkl')
print(json.dumps(modeling_summary['best_model'], indent=2, default=str))

Model tersimpan di models/: model_logistic_regression.pkl, model_svm.pkl, model_naive_bayes.pkl, best_model.pkl
"Logistic Regression"


## Kesimpulan Tahap Modeling

- 3 model dilatih (>= 2 sesuai syarat minimum brief): Logistic Regression, SVM (LinearSVC),
  Multinomial Naive Bayes
- Evaluasi memakai Accuracy, Precision, Recall, F1 (macro) + validasi tambahan 5-fold
  cross-validation agar hasil tidak bergantung pada satu kali split data
- Model terbaik dipilih berdasarkan **Macro F1** (bukan hanya Accuracy) karena lebih adil untuk
  kondisi kelas sentimen yang mungkin tidak seimbang
- Seluruh model disimpan untuk keperluan Tahap 10 (Evaluation mendalam: confusion matrix, error
  analysis) dan Tahap 13 (Deployment Streamlit)

**Status: Tahap 9 (Modeling) selesai.** Lanjut ke **Tahap 10: Model Evaluation** (confusion
matrix per model, analisis kesalahan, pemilihan model final untuk deployment).